# polar_express accuracy check

This notebook unit-tests the current `polar_express` implementation in `train_gpt_leon.py` against a float64 ground-truth matrix inverse-square-root.

Importing `train_gpt_leon.py` directly would initialize distributed training, so this notebook instead:

- parses the current source file,
- extracts `polar_express_coeffs` and the `polar_express` function body,
- imports the real Triton helper kernels from `triton_kernels.py`,
- sanity-checks those kernels against PyTorch matmuls,
- compares the resulting `polar_express` output against an exact float64 reference.

The Triton kernels in this repo are hardcoded around the `K == 768` fast path, so the test cases below keep the reduced dimension at 768.

In [5]:
import ast
import copy
from pathlib import Path

import torch

from triton_kernels import XXT, XTX, ba_plus_cAA

torch.set_printoptions(precision=6, sci_mode=False)

SOURCE_PATH = Path("train_gpt_leon.py")
source = SOURCE_PATH.read_text()
module = ast.parse(source)

polar_express_coeffs = None
polar_express_node = None

for node in module.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id == "polar_express_coeffs":
                polar_express_coeffs = ast.literal_eval(node.value)
    elif isinstance(node, ast.FunctionDef) and node.name == "polar_express":
        polar_express_node = node

assert polar_express_coeffs is not None, "Could not find polar_express_coeffs in train_gpt_leon.py"
assert polar_express_node is not None, "Could not find polar_express in train_gpt_leon.py"

assert torch.cuda.is_available(), "This notebook is meant to exercise the Triton kernels, so it requires CUDA."

device = torch.device("cuda")
print(f"Using device: {device}")
print(f"Loaded {len(polar_express_coeffs)} coefficient triplets from {SOURCE_PATH}")


Using device: cuda
Loaded 5 coefficient triplets from train_gpt_leon.py


In [6]:
def XXT_torch(A, out):
    out.copy_(A @ A.mT)
    return out


def XTX_torch(A, out):
    out.copy_(A.mT @ A)
    return out


def ba_plus_cAA_torch(A, alpha, beta, out):
    out.copy_(beta * A + alpha * (A @ A.mT))
    return out


polar_express_ast = copy.deepcopy(polar_express_node)
polar_express_ast.decorator_list = []
polar_express_module = ast.Module(body=[polar_express_ast], type_ignores=[])
polar_express_src = ast.unparse(ast.fix_missing_locations(polar_express_module))

namespace = {
    "torch": torch,
    "polar_express_coeffs": polar_express_coeffs,
    "XXT": XXT,
    "XTX": XTX,
    "ba_plus_cAA": ba_plus_cAA,
}
exec(polar_express_src, namespace)
polar_express_under_test = namespace["polar_express"]

print(polar_express_src.splitlines()[0])


def polar_express(grad_chunk: torch.Tensor, momentum_buffer: torch.Tensor, second_momentum_buffer: torch.Tensor, momentum_t: torch.Tensor, beta2_t: torch.Tensor, split_baddbmm: bool=False):


In [14]:
def kernel_rel_error(actual, expected):
    actual64 = actual.to(torch.float64)
    expected64 = expected.to(torch.float64)
    return ((actual64 - expected64).norm() / expected64.norm().clamp_min(1e-30)).item()


kernel_results = []

x_wide = torch.randn(2, 48, 768, device=device, dtype=torch.float32)
out_xxt = torch.empty(2, 48, 48, device=device, dtype=torch.float32)
XXT(x_wide, out_xxt)
ref_xxt = XXT_torch(x_wide, torch.empty_like(out_xxt))
kernel_results.append(("XXT fp32", kernel_rel_error(out_xxt, ref_xxt)))

x_tall = torch.randn(2, 896, 768, device=device, dtype=torch.float32)
out_xtx = torch.empty(2, 768, 768, device=device, dtype=torch.float32)
XTX(x_tall, out_xtx)
ref_xtx = XTX_torch(x_tall, torch.empty_like(out_xtx))
kernel_results.append(("XTX fp32", kernel_rel_error(out_xtx, ref_xtx)))

x_wide_bf16 = torch.randn(2, 48, 768, device=device, dtype=torch.float32).bfloat16()
out_xxt_bf16 = torch.empty(2, 48, 48, device=device, dtype=torch.bfloat16)
XXT(x_wide_bf16, out_xxt_bf16)
ref_xxt_bf16 = XXT_torch(x_wide_bf16.float(), torch.empty_like(out_xxt))
kernel_results.append(("XXT bf16", kernel_rel_error(out_xxt_bf16, ref_xxt_bf16)))

x_tall_bf16 = torch.randn(2, 896, 768, device=device, dtype=torch.float32).bfloat16()
out_xtx_bf16 = torch.empty(2, 768, 768, device=device, dtype=torch.bfloat16)
XTX(x_tall_bf16, out_xtx_bf16)
ref_xtx_bf16 = XTX_torch(x_tall_bf16.float(), torch.empty_like(out_xtx))
kernel_results.append(("XTX bf16", kernel_rel_error(out_xtx_bf16, ref_xtx_bf16)))

a_square = torch.randn(3, 48, 48, device=device, dtype=torch.float32)
a_square = 0.5 * (a_square + a_square.mT)
out_ba_plus_caa = torch.empty_like(a_square)
ba_plus_cAA(a_square, alpha=0.37, beta=-1.15, out=out_ba_plus_caa)
ref_ba_plus_caa = ba_plus_cAA_torch(a_square, alpha=0.37, beta=-1.15, out=torch.empty_like(a_square))
kernel_results.append(("ba_plus_cAA fp32", kernel_rel_error(out_ba_plus_caa, ref_ba_plus_caa)))

for name, rel_err in kernel_results:
    print(f"{name:<16} rel_fro={rel_err:.4e}")

assert max(rel_err for _, rel_err in kernel_results) < 2e-2, "Triton kernel mismatch is larger than expected"


XXT fp32         rel_fro=7.0678e-04
XTX fp32         rel_fro=7.3959e-04
XXT bf16         rel_fro=1.5646e-03
XTX bf16         rel_fro=1.4835e-03
ba_plus_cAA fp32 rel_fro=6.5708e-04


The exact target depends on aspect ratio:

- wide matrices (`rows < cols`): `Y = (G G^T + L)^(-1/2) G`
- tall or square matrices (`rows >= cols`): `Y = G (G^T G + L)^(-1/2)`

The tall case is the same matrix function written with the smaller Gram matrix on the right, which is what the implementation does.

In [8]:
def symmetrize(A):
    return 0.5 * (A + A.mT)


def inverse_sqrt_psd(A, eps=1e-12):
    A = symmetrize(A.to(torch.float64))
    evals, evecs = torch.linalg.eigh(A)
    evals = evals.clamp_min(eps)
    inv_sqrt = evecs @ torch.diag_embed(evals.rsqrt()) @ evecs.mT
    return symmetrize(inv_sqrt)


def exact_target(g, L, eps=1e-12):
    g64 = g.to(torch.float64)
    L64 = symmetrize(L.to(torch.float64))
    if g.shape[-2] >= g.shape[-1]:
        return g64 @ inverse_sqrt_psd(g64.mT @ g64 + L64, eps)
    return inverse_sqrt_psd(g64 @ g64.mT + L64, eps) @ g64


def relative_fro_error(actual, expected):
    actual64 = actual.to(torch.float64)
    expected64 = expected.to(torch.float64)
    return ((actual64 - expected64).norm() / expected64.norm().clamp_min(1e-30)).item()


def cosine_similarity(actual, expected):
    a = actual.reshape(-1).to(torch.float64)
    b = expected.reshape(-1).to(torch.float64)
    return (torch.dot(a, b) / (a.norm() * b.norm()).clamp_min(1e-30)).item()


def make_psd(batch_shape, dim, device, seed, scale=1.0):
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)
    basis = torch.randn(*batch_shape, dim, dim, device=device, dtype=torch.float64, generator=gen)
    psd = basis @ basis.mT
    psd = psd / dim
    return (scale * symmetrize(psd)).to(torch.float32)


def run_single_case(
    name,
    shape,
    momentum=0.95,
    beta2=0.8,
    split_baddbmm=False,
    seed=0,
    grad_scale=1.0,
    momentum_scale=0.25,
    l_scale=0.5,
):
    gen = torch.Generator(device=device)
    gen.manual_seed(seed)

    grad = grad_scale * torch.randn(*shape, device=device, dtype=torch.float32, generator=gen)
    momentum_buffer = momentum_scale * torch.randn(*shape, device=device, dtype=torch.float32, generator=gen)

    rows, cols = shape[-2:]
    gram_dim = cols if rows >= cols else rows
    second_momentum_buffer = make_psd(shape[:-2], gram_dim, device, seed + 1, scale=l_scale)

    grad_in = grad.clone()
    momentum_in = momentum_buffer.clone()
    second_momentum_in = second_momentum_buffer.clone()

    out = polar_express_under_test(
        grad_in,
        momentum_in,
        second_momentum_in,
        torch.tensor(momentum),
        torch.tensor(beta2),
        split_baddbmm=split_baddbmm,
    )

    g_after = grad_in.clone()
    l_after = second_momentum_in.clone()
    exact = exact_target(g_after, l_after)

    metrics = {
        "name": name,
        "shape": tuple(shape),
        "branch": "tall/right" if rows >= cols else "wide/left",
        "split_baddbmm": split_baddbmm,
        "rel_fro": relative_fro_error(out, exact),
        "max_abs": (out.to(torch.float64) - exact).abs().max().item(),
        "cosine": cosine_similarity(out, exact),
    }
    return metrics, out, exact, g_after, l_after


def print_metrics_table(results):
    header = f"{'case':<16} {'shape':<16} {'branch':<12} {'split':<6} {'rel_fro':>10} {'max_abs':>10} {'cosine':>10}"
    print(header)
    print("-" * len(header))
    for row in results:
        print(
            f"{row['name']:<16} {str(row['shape']):<16} {row['branch']:<12} "
            f"{str(row['split_baddbmm']):<6} {row['rel_fro']:>10.4e} {row['max_abs']:>10.4e} {row['cosine']:>10.6f}"
        )


In [15]:
cases = [
    {"name": "wide_2d", "shape": (48, 768), "seed": 1, "split_baddbmm": False},
    {"name": "tall_2d", "shape": (896, 768), "seed": 2, "split_baddbmm": False},
    {"name": "square_2d", "shape": (768, 768), "seed": 3, "split_baddbmm": False},
    {"name": "wide_batched", "shape": (2, 48, 768), "seed": 4, "split_baddbmm": True},
    {"name": "tall_batched", "shape": (2, 896, 768), "seed": 5, "split_baddbmm": True},
]

results = []
for case in cases:
    metrics, *_ = run_single_case(**case)
    results.append(metrics)

print_metrics_table(results)

max_rel_fro = max(row["rel_fro"] for row in results)
min_cosine = min(row["cosine"] for row in results)

print()
print(f"max relative Frobenius error: {max_rel_fro:.4e}")
print(f"min cosine similarity:       {min_cosine:.6f}")

assert max_rel_fro < 1.1e-1, "Relative error is larger than the validated envelope for the current coeffs and dtypes"
assert min_cosine > 0.994, "Direction drift is larger than the validated envelope"


case             shape            branch       split     rel_fro    max_abs     cosine
--------------------------------------------------------------------------------------
wide_2d          (48, 768)        wide/left    False  9.1760e-02 7.2940e-03   0.996041
tall_2d          (896, 768)       tall/right   False  9.7158e-02 8.9618e-03   0.995304
square_2d        (768, 768)       tall/right   False  9.7062e-02 1.0171e-02   0.995346
wide_batched     (2, 48, 768)     wide/left    True   1.0376e-01 9.0378e-03   0.994807
tall_batched     (2, 896, 768)    tall/right   True   9.8113e-02 1.0076e-02   0.995214

max relative Frobenius error: 1.0376e-01
min cosine similarity:       0.994807
